In [1]:
pip install pandas matplotlib seaborn python-pptx openpyxl

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.



In [7]:
# merge_incidents_fixed.py
import pandas as pd
from pathlib import Path
import sys

# filenames (change paths if needed)
FILE_2023 = "incident-notifications-received-2023.xlsx"
FILE_2024 = "incident-notifications-received-2024.xlsx"
OUT_DIR = Path("output")
OUT_DIR.mkdir(exist_ok=True)

def read_excel_first_sheet(path):
    try:
        df = pd.read_excel(path, engine="openpyxl")
        print(f"Read {path} shape: {df.shape}")
        return df
    except FileNotFoundError:
        print(f"ERROR: file not found: {path}", file=sys.stderr)
        raise
    except Exception as e:
        print(f"ERROR reading {path}: {e}", file=sys.stderr)
        raise

def normalize_columns(df):
    df = df.copy()
    df.columns = [str(c).strip().lower().replace(" ", "_").replace("-", "_") for c in df.columns]
    return df

# --- Read files
df23 = read_excel_first_sheet(FILE_2023)
df24 = read_excel_first_sheet(FILE_2024)

# --- Normalize column names
df23 = normalize_columns(df23)
df24 = normalize_columns(df24)

print("\n2023 cols:", df23.columns.tolist())
print("\n2024 cols:", df24.columns.tolist())

# --- Explicit mapping based on your files
# Use the exact names you reported; fallback to close alternatives if needed.
date_col = None
for c in ["received","incident_date","date_received","date"]:
    if c in df23.columns or c in df24.columns:
        date_col = c
        break

product_col = None
for c in ["producttype","product_type","product","product_category","productdescription"]:
    if c in df23.columns or c in df24.columns:
        product_col = c
        break

origin_col = None
for c in ["origincountry","origin_country","country_of_origin","country"]:
    if c in df23.columns or c in df24.columns:
        origin_col = c
        break

notifier_col = None
for c in ["notifiertype","notifier_type","notified_by","source","reporter"]:
    if c in df23.columns or c in df24.columns:
        notifier_col = c
        break

type_col = None
for c in ["incidenttype","incident_type","notification_type","type"]:
    if c in df23.columns or c in df24.columns:
        type_col = c
        break

ref_col = None
for c in ["incident_id","reference","notification_id","id"]:
    if c in df23.columns or c in df24.columns:
        ref_col = c
        break

print("\nDetected mapping:")
print(" date_col:", date_col)
print(" product_col:", product_col)
print(" origin_col:", origin_col)
print(" notifier_col:", notifier_col)
print(" type_col:", type_col)
print(" ref_col:", ref_col)

# Build keep list = union of the useful columns that exist
useful_candidates = [ref_col, date_col, type_col, product_col, notifier_col, origin_col,
                     "incidentsource", "postcode", "local_authority", "business_name", "business_type",
                     "latitude","longitude"]
keep = [c for c in useful_candidates if c and (c in df23.columns or c in df24.columns)]

# Also include any column that is common to both if keep is empty
if not keep:
    common = set(df23.columns).intersection(set(df24.columns))
    keep = sorted(list(common))

print("\nColumns to keep:", keep)

# Subset safely (only columns actually present)
df23_sub = df23[[c for c in keep if c in df23.columns]].copy()
df24_sub = df24[[c for c in keep if c in df24.columns]].copy()

# Mark source
df23_sub["source_file_year"] = 2023
df24_sub["source_file_year"] = 2024

# Concatenate
merged = pd.concat([df23_sub, df24_sub], axis=0, ignore_index=True, sort=False)
print("\nMerged shape before cleaning:", merged.shape)

# Trim strings on object columns
for c in merged.select_dtypes(include=["object"]).columns:
    merged[c] = merged[c].astype(str).str.strip()

# Parse date
if date_col and date_col in merged.columns:
    merged["incident_date"] = pd.to_datetime(merged[date_col], errors="coerce", dayfirst=True)
else:
    # attempt any column containing 'date'
    date_candidates = [c for c in merged.columns if "date" in c]
    if date_candidates:
        merged["incident_date"] = pd.to_datetime(merged[date_candidates[0]], errors="coerce", dayfirst=True)
    else:
        merged["incident_date"] = pd.NaT

# Year and month helpers
merged["year"] = merged["incident_date"].dt.year.fillna(merged.get("source_file_year", pd.NA)).astype("Int64")
merged["month"] = merged["incident_date"].dt.to_period("M").astype(str).fillna("")

# Safe creation of cleaned product_type column
if product_col and product_col in merged.columns:
    merged["product_type_clean"] = merged[product_col].astype(str).str.lower().str.replace(r"\s+", " ", regex=True)
else:
    # fallback: look for any column with 'product' in name
    prod_cols = [c for c in merged.columns if "product" in c]
    if prod_cols:
        merged["product_type_clean"] = merged[prod_cols[0]].astype(str).str.lower().str.replace(r"\s+", " ", regex=True)
    else:
        merged["product_type_clean"] = ""

# Safe creation of origin_country_clean
if origin_col and origin_col in merged.columns:
    merged["origin_country_clean"] = merged[origin_col].astype(str).fillna("").str.strip()
else:
    cand = [c for c in merged.columns if "country" in c]
    merged["origin_country_clean"] = merged[cand[0]].astype(str).fillna("").str.strip() if cand else ""

# notifier
if notifier_col and notifier_col in merged.columns:
    merged["notifier_type_clean"] = merged[notifier_col].astype(str).fillna("").str.strip()
else:
    cand = [c for c in merged.columns if "notifier" in c or "notified" in c or "source" in c]
    merged["notifier_type_clean"] = merged[cand[0]].astype(str).fillna("").str.strip() if cand else ""

# incident type
if type_col and type_col in merged.columns:
    merged["incident_type_clean"] = merged[type_col].astype(str).fillna("").str.strip()
else:
    cand = [c for c in merged.columns if "incident" in c or "notification" in c or "type" == c]
    merged["incident_type_clean"] = merged[cand[0]].astype(str).fillna("").str.strip() if cand else ""

# Drop exact duplicates
before = len(merged)
merged = merged.drop_duplicates()
after = len(merged)
print(f"\nDropped {before-after} exact duplicate rows; remaining rows: {after}")

# Save CSV
OUT_DIR = Path(r"C:\Users\HP\OneDrive - Leeds Trinity University\MS in Data Science and AI\Data Science and Visualisation\Assignment")
OUT_DIR.mkdir(exist_ok=True)
merged.to_csv(OUT_DIR / "merged_incidents.csv", index=False)
print(f"\nMerged CSV written to: {out_csv}")

# Preview few rows
print("\nPreview:")
print(merged.head(8).to_string(index=False))

print("\nDone. Please open 'output/merged_incidents.csv' to inspect before loading into Power BI.")


Read incident-notifications-received-2023.xlsx shape: (1822, 7)
Read incident-notifications-received-2024.xlsx shape: (1766, 7)

2023 cols: ['incident_id', 'received', 'incidenttype', 'producttype', 'notifiertype', 'origincountry', 'incidentsource']

2024 cols: ['incident_id', 'received', 'incidenttype', 'producttype', 'notifiertype', 'origin_country', 'incidentsource']

Detected mapping:
 date_col: received
 product_col: producttype
 origin_col: origincountry
 notifier_col: notifiertype
 type_col: incidenttype
 ref_col: incident_id

Columns to keep: ['incident_id', 'received', 'incidenttype', 'producttype', 'notifiertype', 'origincountry', 'incidentsource']

Merged shape before cleaning: (3588, 8)

Dropped 0 exact duplicate rows; remaining rows: 3588

Merged CSV written to: output\merged_incidents.csv

Preview:
incident_id            received                              incidenttype                               producttype                                                     notifier

In [10]:
pip install pandas python-dateutil openpyxl

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.



In [21]:
# final_export_seven_cols.py
import pandas as pd
import numpy as np
from pathlib import Path
from dateutil import parser
import re
import sys

# ----------------- CONFIG: update if you moved folder -----------------
BASE = Path(r"C:\Users\HP\OneDrive - Leeds Trinity University\MS in Data Science and AI\Data Science and Visualisation\Assignment")
FILE_2023 = BASE / "incident-notifications-received-2023.xlsx"
FILE_2024 = BASE / "incident-notifications-received-2024.xlsx"
OUT_FILE = BASE / "merged_incidents_final_clean.csv"
# ---------------------------------------------------------------------

# Robust date parse that returns date only (no time)
def parse_date_to_dateonly(x):
    if pd.isna(x):
        return pd.NaT
    s = str(x).strip()
    if s == "" or s.lower() in {"nan","none","null","n/a","unknown","-"}:
        return pd.NaT
    # remove common prefixes
    s = re.sub(r'(?i)received[:\s-]*', '', s).strip()
    # remove trailing time like " 0:00" or " 00:00:00" or "T00:00:00Z"
    s = re.sub(r'\s+\d{1,2}:\d{2}(:\d{2})?$', '', s)
    s = s.replace("T00:00:00Z", "").replace("T00:00:00", "")
    # try pandas parse first (dayfirst=True for UK)
    try:
        ts = pd.to_datetime(s, dayfirst=True, errors='coerce')
        if pd.notna(ts):
            return ts.date()  # convert to date (YYYY-MM-DD)
    except Exception:
        pass
    # fallback: try dateutil
    try:
        ts = parser.parse(s, dayfirst=True, fuzzy=True)
        return ts.date()
    except Exception:
        return pd.NaT

# load sheets safely
def load_sheet(path):
    if not path.exists():
        print(f"ERROR: file not found: {path}", file=sys.stderr)
        return pd.DataFrame()
    df = pd.read_excel(path, engine="openpyxl")
    # standardize column names to lowercase single tokens for mapping
    df.columns = [str(c).strip() for c in df.columns]
    return df

# map any column variants to canonical target name
def map_to_canonical(df):
    # canonical target names we will produce (internal lower-case keys)
    canonical = {
        "incident_id": ["incident_id","incidentid","id","notification_id","reference","notification_reference"],
        "received": ["received","received_raw","date","date_received","received_raw_clean","received_raw_cleaned"],
        "incidenttype": ["incidenttype","incident_type","notification_type","type","incidenttype_clean","incident_type_clean"],
        "producttype": ["producttype","product_type","product","product_category","producttype_clean","product_type_clean"],
        "notifiertype": ["notifiertype","notifier_type","notified_by","source","notifier","notifiertype_clean","notifier_type_clean"],
        "origin_country": ["origin_country","origincountry","country_of_origin","country","origin_country_clean"],
        "incidentsource": ["incidentsource","incident_source","source","incidentsource_clean"]
    }
    out = pd.DataFrame()
    cols_lower = {c.lower().replace(" ","").replace("_",""): c for c in df.columns}
    # for each canonical key, try to find first matching actual column name
    for target, variants in canonical.items():
        found = None
        for v in variants:
            # compare normalized forms
            key = v.lower().replace(" ","").replace("_","")
            if key in cols_lower:
                found = cols_lower[key]
                break
        if found:
            out[target] = df[found]
        else:
            # not found → create empty column
            out[target] = [np.nan]*len(df)
    return out

# --- Load both files ---
df23 = load_sheet(FILE_2023)
df24 = load_sheet(FILE_2024)

if df23.empty and df24.empty:
    print("No input files found. Please check paths.")
    sys.exit(1)

# --- Map to canonical for each year df ---
m23 = map_to_canonical(df23) if not df23.empty else pd.DataFrame(columns=list(map_to_canonical(df24).columns))
m24 = map_to_canonical(df24) if not df24.empty else pd.DataFrame(columns=list(map_to_canonical(df23).columns))

# add year tags (use existing if present else 2023/2024)
m23["year_source_file"] = 2023
m24["year_source_file"] = 2024

# --- Concat ---
merged = pd.concat([m23, m24], ignore_index=True, sort=False)

# --- Parse and overwrite Received into date-only, then rename
merged["incident_date_parsed"] = merged["received"].apply(parse_date_to_dateonly)

# Create final column names exactly as you requested:
# Incident ID, Received (date), IncidentType, ProductType, NotifierType, Origin Country, IncidentSource

# Prepare values (strip strings, replace dummy text with NaN)
def clean_text_col(series):
    return series.astype(str).str.strip().replace({"nan":np.nan,"None":np.nan,"none":np.nan,"NULL":np.nan,"null":np.nan})

merged["Incident ID"] = clean_text_col(merged["incident_id"])
# Received as date only in format YYYY-MM-DD (ISO). If you prefer DD/MM/YYYY as string, uncomment the formatting below.
merged["Received"] = merged["incident_date_parsed"]  # pandas objects (date) → will be ISO when saved
# If you want DD/MM/YYYY strings instead of ISO date, uncomment:
# merged["Received"] = merged["incident_date_parsed"].apply(lambda d: d.strftime("%d/%m/%Y") if pd.notna(d) else "")

merged["IncidentType"] = clean_text_col(merged["incidenttype"])
merged["ProductType"] = clean_text_col(merged["producttype"])
merged["NotifierType"] = clean_text_col(merged["notifiertype"])
merged["Origin Country"] = clean_text_col(merged["origin_country"])
merged["IncidentSource"] = clean_text_col(merged["incidentsource"])

# --- Keep only these exact seven columns (in this order) ---
final_cols = ["Incident ID","Received","IncidentType","ProductType","NotifierType","Origin Country","IncidentSource"]
final = merged[final_cols].copy()

# --- Remove exact duplicates by Incident ID, keeping the row with the latest Received date (if available) ---
# If Incident ID is NaN, keep unique rows (dedupe fully)
if final["Incident ID"].notna().sum() > 0:
    # convert Received to datetime for sorting (if it isn't)
    final["__received_dt"] = pd.to_datetime(final["Received"], errors="coerce")
    final = final.sort_values(by=["Incident ID","__received_dt"], ascending=[True, False])
    final = final.drop_duplicates(subset=["Incident ID"], keep="first")
    final = final.drop(columns="__received_dt")
else:
    # no incident ids present → simply drop exact duplicate rows
    before = len(final)
    final = final.drop_duplicates()
    # print how many removed
    print(f"Dropped {before - len(final)} exact duplicate rows (no incident IDs present)")

# --- Final tidy: trim whitespace and replace string 'nan' with empty ---
for c in final.columns:
    if final[c].dtype == object:
        final[c] = final[c].astype(str).str.strip().replace({"nan":"","None":""})

# --- Save output CSV ---
final.to_csv(OUT_FILE, index=False, encoding="utf-8", date_format="%Y-%m-%d")
print("Saved final cleaned file with exactly 7 columns at:")
print(OUT_FILE)
print("Final rows,cols:", final.shape)
print("Sample rows:")
print(final.head(8).to_string(index=False))


C:\Users\HP\AppData\Local\Temp\ipykernel_18156\1138788191.py:30: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  ts = pd.to_datetime(s, dayfirst=True, errors='coerce')


Saved final cleaned file with exactly 7 columns at:
C:\Users\HP\OneDrive - Leeds Trinity University\MS in Data Science and AI\Data Science and Visualisation\Assignment\merged_incidents_final_clean.csv
Final rows,cols: (3588, 7)
Sample rows:
                                         Incident ID   Received                              IncidentType                               ProductType                                                     NotifierType   Origin Country                IncidentSource
Applied filters:\nYear is 2023\nDublicate is (Blank)        NaT                                                                                                                                                                                                    
                                           I-004-951 2023-01-03                Pathogenic Micro-Organisms                      Fish & Fish Products                                                  Local Authority           Latvia Microbiol